# Generalization Test — do the ~99.9% numbers hold?
### SEA 820 · GenDetect · Classical baseline

The baseline scores ~99.9% macro-F1 **on its own test set**, but the shortcut analysis (step 10b
of `train_baseline.ipynb`) argued that score rides on *dataset artifacts* — placeholder tokens,
prompt-topic words, formatting. The honest way to check is to feed the saved model text it has
**never seen the distribution of**: a modern chatbot answer, a news paragraph, casual writing,
or your own PDF — and watch whether accuracy and confidence survive.

> Uses the models saved by `train_baseline.ipynb` in Drive `classical_results/models/`.

## 0. Setup — load the saved models

In [1]:
# Works LOCALLY (repo layout) and on Colab (Drive) — auto-detects where the models are.
import os, sys, numpy as np, joblib

_here = os.getcwd()
if os.path.isdir(os.path.join(_here, 'classical_results', 'models')):
    CM_DIR = _here                                        # kernel started in classical_model/
elif os.path.isdir(os.path.join(_here, 'classical_model', 'classical_results', 'models')):
    CM_DIR = os.path.join(_here, 'classical_model')       # kernel started at repo root
elif os.path.isdir('/content/drive/MyDrive/NLP_project/classical_results/models'):
    from google.colab import drive; drive.mount('/content/drive')
    CM_DIR = '/content/drive/MyDrive/NLP_project'         # Colab / Drive
else:
    raise FileNotFoundError('classical_results/models not found — set CM_DIR by hand.')

MODELS_DIR = os.path.join(CM_DIR, 'classical_results', 'models')
DATA_DIR   = os.path.normpath(os.path.join(CM_DIR, os.pardir, 'data_processing'))


for p in (CM_DIR, DATA_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

import tfidf_features as tf
from preprocessing import classical_preprocess          # SAME cleaning used at training time

vec    = joblib.load(os.path.join(MODELS_DIR, 'tfidf_vectorizer.joblib'))
config = joblib.load(os.path.join(MODELS_DIR, 'config.joblib'))
models = {
    'LogReg':    joblib.load(os.path.join(MODELS_DIR, 'LogReg.joblib')),
    'LinearSVC': joblib.load(os.path.join(MODELS_DIR, 'LinearSVC.joblib')),
}
print('loaded from :', MODELS_DIR)
print('models      :', list(models.keys()))
print('config      :', config)

c:\Users\sakib\Desktop\semester 8\NLP\final_project\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loaded from : c:\Users\sakib\Desktop\semester 8\NLP\final_project\classical_model\classical_results\models
models      : ['LogReg', 'LinearSVC']
config      : {'best_ngram': (1, 2), 'best_maxf': 50000, 'best_C_LR': 10.0, 'best_C_SVM': 10.0, 'use_length': True, 'best_model': 'LinearSVC'}


c:\Users\sakib\Desktop\semester 8\NLP\final_project\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\sakib\Desktop\semester 8\NLP\final_project\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\sakib\Desktop\semester 8\NLP\final_project\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to un

## 1. Prediction helper — labels **and** an AI/human %

**What is a `decision_score`?** A linear model computes `score = w·x + b` — the signed distance
from the decision boundary (the separating hyperplane). `score > 0` -> predict **AI**, `< 0` ->
**human**, and the magnitude is roughly how far from the boundary the text sits (a confidence
proxy). But it is **unbounded** (any real number), so it is *not* a probability.

**Showing "60% AI / 40% human" like online detectors — yes, we can:**
- **Logistic Regression** is probabilistic: its score is a *log-odds*, and `predict_proba`
  squashes it through a **sigmoid** into `P(AI)` / `P(human)`. This is the honest percentage.
- **LinearSVC** has no native probability; we show a *sigmoid of its margin* as a rough
  (uncalibrated) approximation — for a proper % you'd wrap it in `CalibratedClassifierCV`.

We still apply the **same `classical_preprocess`** + fitted vectorizer used at training time.

> ⚠️ **Calibration caveat:** this dataset is trivially separable, so the model is
> **over-confident** — expect ~99% one way and rarely a nuanced 60/40. A middle percentage is
> *not* a reliable "how AI is this" meter; it's another symptom of the shortcut.

In [2]:
def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def _featurize(texts):
    cleaned = [classical_preprocess(t, method='lemmatize') for t in texts]
    X = vec.transform(cleaned)
    if config.get('use_length'):
        X = tf.stack_length(X, list(texts))     # match training feature layout
    return X

def predict_text(text, show=True):
    """Return {model: {pred, p_ai, calibrated}} and optionally pretty-print AI/human %."""
    X = _featurize([text])
    res = {}
    for name, mdl in models.items():
        pred = int(mdl.predict(X)[0])                       # 1 = AI, 0 = human
        if hasattr(mdl, 'predict_proba'):                  # LogReg -> real probability
            p_ai, calibrated = float(mdl.predict_proba(X)[0][1]), True
        else:                                              # LinearSVC -> sigmoid(margin), rough
            p_ai, calibrated = float(_sigmoid(mdl.decision_function(X)[0])), False
        res[name] = {'pred': 'AI' if pred else 'human', 'p_ai': p_ai, 'calibrated': calibrated}
    if show:
        print(f'[{len(text.split()):>4} words] "{text[:80].strip()}..."')
        for name, r in res.items():
            tag = '' if r['calibrated'] else '   (uncalibrated approx)'
            print(f'    {name:10s} -> {r["pred"]:5s} | '
                  f'AI {r["p_ai"]*100:5.1f}%  /  human {(1-r["p_ai"])*100:5.1f}%{tag}')
        print()
    return res

## 3. Generalization test — OUT-OF-DISTRIBUTION text
A small hand-labeled set from *outside* the PERSUADE-essay / AI-essay distribution: casual
writing, a news-style paragraph, a modern chatbot answer, a marketing blurb. If the ~99.9% were
real "AI detection", accuracy here should stay high. If it was a shortcut, expect it to drop and
the confidence to stay stubbornly high (over-confident + wrong).

In [3]:
# (text, true_label)  -- 0 = human, 1 = AI-generated
OOD = [
    # --- human, but NOT student essays ---
    ("I don't know where to start this so i'm just going to word vomit: I’m currently 17F, around 6 months ago my male coworker asked me on a date, I went and had a great time. He told me he was 21 turning 22, immediately I was a little bit uncomfortable but since the age of consent where I live is 16 he said it was completely legal.", 0),
    ("The night I found out his real age we went out and he wanted me to meet his “friend”, the women he introduced me to was in her mid 30s😭 I questioned why a 21 year old male would be friends with a 36 year old women and he got mad, I questioned him and asked him to show me his id. He refused and said “i’ll show you when we get home” The whole car ride home was extremely uncomfortable and scary. I had to force the id out of his hand and my fear became true.", 0),
    
    # --- AI-generated, from a modern chatbot (not the training generator) ---
    ("Firefox supports the exact same Developer Console script as Chrome. This runs natively in Firefox without needing to download anything extra:", 1),
    ("I thrive on exploring the bleeding edge of technology, particularly the intersection of intuitive UI—what I consider strong product taste—and applied AI, including agentic engineering, large language models, and the Model Context Protocol (MCP). Whether I am architecting multi-agent systems in Python and TypeScript or rapidly learning a new tool to prototype a solution, my focus remains on how these innovations can be leveraged to automate workflows and seamlessly improve operations.", 1),
    ("Please provide a detailed analysis of the current state of the economy and its potential impact on the job market.", 1),
]

correct = {name: 0 for name in models}
for text, true in OOD:
    res = predict_text(text, show=True)
    truth = 'AI' if true else 'human'
    for name, r in res.items():
        hit = (r['pred'] == truth)
        correct[name] += hit
    print(f'    (true label: {truth})')
    print('    ' + '-'*44)

print('\nAccuracy on this out-of-distribution set:')
for name in models:
    print(f'   {name:10s}: {correct[name]}/{len(OOD)} = {correct[name]/len(OOD):.0%}')
print(f'\nCompare with ~99.9% on the in-distribution test set. That gap IS the caveat.')

[  68 words] "I don't know where to start this so i'm just going to word vomit: I’m currently..."
    LogReg     -> human | AI  48.2%  /  human  51.8%
    LinearSVC  -> human | AI  46.0%  /  human  54.0%   (uncalibrated approx)

    (true label: human)
    --------------------------------------------
[  98 words] "The night I found out his real age we went out and he wanted me to meet his “fri..."
    LogReg     -> AI    | AI  53.6%  /  human  46.4%
    LinearSVC  -> AI    | AI  52.3%  /  human  47.7%   (uncalibrated approx)

    (true label: human)
    --------------------------------------------
[  21 words] "Firefox supports the exact same Developer Console script as Chrome. This runs na..."
    LogReg     -> AI    | AI 100.0%  /  human   0.0%
    LinearSVC  -> AI    | AI  87.5%  /  human  12.5%   (uncalibrated approx)

    (true label: AI)
    --------------------------------------------
[  71 words] "I thrive on exploring the bleeding edge of technology, particularly the intersec.

### What the gap means (fill in after running)
- **In-distribution test:** ~99.9% macro-F1.
- **Out-of-distribution set above:** ____% / ____%.
- **Pattern:** the model tends to call ____ text "AI" (formal vocabulary / structure) and can miss
  ____ AI text that lacks the training artifacts.
- **Why:** it learned *source/topic tells* (PERSUADE prompts, redaction placeholders, essay
  connectives), not general "AI-ness" — so it doesn't transfer.
- **Takeaway for the report:** the baseline number is real but **non-transferable**; a fair
  claim is "high on this dataset, unproven out-of-domain." This is also why DistilBERT beating it
  on the same test set won't, by itself, prove better *generalization*.

## 4. Try your own document — paste text

In [4]:
my_text = """

Algorithmic Determinism and the Quantum Aesthetic: How Editing Constructs Meaning in Tom Tykwer’s Run Lola RunFilm Specifications and Narrative ArchitectureTitle: Run Lola Run (Original German Title: Lola rennt)   Director/Screenplay: Tom Tykwer   Editor: Mathilde BonnefoyPrincipal Cast: Franka Potente (Lola), Moritz Bleibtreu (Manni), Herbert Knaup (Lola's Father)   Genre: Experimental Techno-Thriller / Avant-Garde Drama   Year of Production: 1998Plot SynopsisSet within the frantic urban landscape of post-unification Berlin, Run Lola Run follows Lola, a young woman who receives an agitated telephone call from her boyfriend, Manni. Manni, working as a courier for a criminal enterprise, has inadvertently left a bag containing 100,000 Deutsche Marks on a subway train, where it was subsequently claimed by a transient. Facing execution by his employer in exactly twenty minutes, Manni relies on Lola to secure the replacement capital and reach him before the deadline expire. The film structurally unfolds across three distinct twenty-minute iterations, with each "run" initiated by minor spatial and behavioral deviations that radically redirect the characters' destinies.  The Ideological Matrix: Free Will, Determinism, and the Ethics of Quantum PossibilityAt its intellectual core, Run Lola Run operates as an elegant cinematic treatise on the tension between absolute determinism and radical free will. Tykwer utilizes a cyclical, multi-scenario framework to interrogate the extent to which human agents are genuinely free, or whether they are simply buffeted about by the choices of other willing entities and rigid physical constraints. The film constructs what can be termed the "ethics of quantum possibility". Under this paradigm, every micro-decision made by an individual possesses a staggering moral weight; every moment matters, and choices ripple outward to intersect with other human lives in entirely unpredictable, non-linear trajectories.  This philosophical framing is explicitly established via the movie's introductory paratexts and quotations. Tykwer first invokes lines from the final section of T.S. Eliot’s Four Quartets, "Little Gidding":  "We shall not cease from exploration, and the end of all our exploring will be to arrive where we started and know the place for the first time."   This literary excerpt serves as an overarching thematic map for the narrative architecture, establishing a loop devoid of conventional linear time. It underscores that deep wisdom and clarity are won solely through cumulative experience and systemic failure; Lola retains an implicit baseline of memory from her past runs, carrying lessons forward to optimize her subsequent paths.  Figure 1: The Eliot Learning Loop Flowchart[T.S. Eliot Theory] ---> Cumulative Failure ---> Experience ---> The Final Run
     ^                                                                |
     +----------------- Temporal Iteration Feedback Loop -------------+
Directly following Eliot, Tykwer introduces a legendary aphorism from German football player Sepp Herberger: "After the game is before the game" and "The ball is round, the game lasts 90 minutes. Everything else is pure theory.". This abrupt tonal shift grounds the film's high-minded metaphysics into concrete, rule-bound systems.  Human beings are shown to be embodied, finite creatures trapped within the structural constraints of biology, physical geography, and technological mediums. Free will is not an unconstrained void; rather, it is the capacity to execute conscious choices within the rigid boundaries imposed by external realities. The narrator emphasizes this existential condition by noting that while humanity faces an internal regress of endless queries, we ultimately spend our lives seeking an answer to a single fundamental question concerning purpose and destiny.  Structural Editing: The Branching Logic of the Parallel MultiverseTo visually articulate this complex web of causality and quantum potentiality, the film rejects conventional Hollywood continuity cutting in favor of an overt, highly aggressive montage style. Editing is not merely a tool for narrative delivery in Run Lola Run; it is the foundational mechanism that generates the film's deeper meaning. The structural blueprint of the editing relies on three distinct sequences that feature identical starting shots but yield vastly divergent outcomes. This technique mimics an algorithmic decision tree or a dynamic digital mind map. The city landscape is transformed into an isometric obstacle course where everyday urban fixtures—streets, cars, buildings, and pedestrians—act as physical variables designed to delay or accelerate Lola's progress.  Figure 2: The Algorithmic Multiverse Timeline                     +---> Run 1: Suboptimal Paths (Catastrophic Collision / Death)
                     |
[Identical Start] ---+---> Run 2: Behavioral Adjustment (Rebellion / Bank Robbery)
                     |
                     +---> Run 3: Algorithmic Serendipity (Casino Mastery / Equilibrium)
This structural architecture aligns seamlessly with video game mechanics, functioning as a lived manifestation of interactive logic. In the initial run, Lola makes suboptimal choices, encounters friction, and suffers terminal failure via a catastrophic collision with destiny. Rather than terminating the narrative permanently, the editing engine executes a hard "restart". As the second and third runs commence, Lola retains a form of implicit systemic muscle memory: she adapts her interaction with a staircase dog, masters the handling of a firearm, and alters her precise spatial negotiation with a passing ambulance. Bonnefoy's editing treats these alternate realities not as mere stylistic gimmicks, but as parallel domains that demonstrate how minuscule adjustments in execution yield entirely different macro-scenarios.  The Micro-Mechanics of Editing: Photo Montages and Flash-ForwardsOne of the most innovative applications of editing within Run Lola Run is its treatment of minor secondary characters. As Lola sprints through the public squares of Berlin, she briefly collides with or passes various strangers. The editing immediately ruptures the primary timeline by inserting ultra-rapid, hyper-accelerated photographic montages that catalog the potential future lives of these random individuals.  Secondary Character Trajectory MatrixWoman Pushing Baby Carriage: * Run 1: Destitution; child is legally seized, leading her to abduct another.Run 2: Wins the lottery; sudden material affluence.Run 3: Undergoes a spiritual awakening, finding salvation as a Jehovah's Witness.Boy on Stolen Bicycle: * Run 1: Becomes a marginalized vagrant after a series of misfortunes.Run 2: Suffers a horrific, debilitating road accident.Run 3: Sells the bicycle to Manni’s homeless man, altering the recovery of the cash.Bank Corridor Clerk: * Run 1: Remains trapped in a cycle of clinical depression and professional stagnation.Run 2: Involved in a sudden, violent workplace confrontation.Run 3: Achieves a quiet, stable domestic contentment.These structural interruptions are systematically anchored by an acoustic cue: the piercing, mechanical whine of a camera capacitor powering up for a flash. This editing strategy offers a series of brief snapshots of alternative futures dictated entirely by the precise millisecond Lola intersects with them. It visually proves the assertion that even within a massive, blurred rush of humanity, single micro-actions produce severe, cascading historical consequences.  Crucially, when the film arrives at its final montage tracking Lola and Manni, their specific future is deliberately withheld from the viewer. By dropping the curtain before their destiny is printed, Bonnefoy’s editing implies that fate is not a pre-rendered loop; it remains an open, unwritten equation left entirely to active human agency.  This thematic concept is underscored by the bedroom interludes positioned between the primary runs. These intimate sequences are edited with an intense, monochromatic red tint and high-contrast lighting. This visual choice transforms the bedroom into a symbolic cinematographic "darkroom," where Lola and Manni analyze the failures of their past trajectories and actively "develop" the existential blueprints for the remainder of their lives.  Furthermore, Tykwer intentionally shifts film stocks throughout these sequences, alternating between grainy, low-fidelity video tracking and pristine, high-definition cinematic film. These shifts in visual texture alert the audience to transitions between rigid, deterministic reality and fluid, psychological spaces of pure imagination and choice.  Theoretical Intersections: From Dialectical Collision to Cognitive OmnipresenceTo contextualize the historical and aesthetic weight of Bonnefoy and Tykwer's editorial methodology, one must view it through the lens of classical and contemporary film theory. David Bordwell and Kristin Thompson define editing in a dual sense: as the physical task of selecting and joining takes, and as the overarching system of techniques governing inter-shot relationships. In Run Lola Run, this relationship is pushed to a modernist extreme that actively fractures the normal space-time continuum described by German critic Rudolph Arnheim. Arnheim observed that while real life features no sudden spatial or temporal jerks, film editing possesses the radical liberty to interrupt continuous axes, joining disconnected spaces to construct a purely synthetic reality.  Figure 3: Space-Time Continuity Comparison[Real Life] = UNINTERRUPTED CONTINUITY AXIS (NO SPATIAL OR TEMPORAL JERKS)
                     vs.
[Run Lola Run] = [Shot Frame A] | [Jump Cut] | [Animation] | [Montage Fragment]
This synthetic disruption is precisely what Maya Deren champions as the "creative use of reality". Deren notes that editing generates entirely new sequential relationships, liberating film from the earthbound, step-by-step causal logic of the nineteenth century. Tykwer’s editing does not seek to document the world passively; it behaves as an active, manipulative entity that mirrors the rapid-fire internal structures of human cognition.  This directly fulfills Hugo Münsterberg’s classic 1916 psychological theory of cinema. Münsterberg argued that the power of film lies in its ability to objectify human mental functions—such as memory, attention, and imagination—directly onto the screen. The abrupt flash-forwards and looping structures in Run Lola Run behave "just as a hasty thought of bygone days darts through the mind," bypassing physical limits to award the spectator a state of total cognitive omnipresence across multiple concurrent timelines.  Furthermore, the relentless momentum of the editing honors the cognitive demands outlined by Noël Carroll. Carroll notes that human beings are naturally wired to shift attention across their environment unless a sudden visual alteration keeps them focused. The rapid straight cuts in Run Lola Run keep the frame constantly vibrating with visual activity, perfectly matching our perceptual systems.  However, Tykwer infuses this seamless cognitive flow with elements of Sergei Eisenstein’s theory of dialectical montage. Eisenstein asserted that editing shouldn't be a smooth linkage of parts, but a violent collision and conflict between opposing shots. By clashing animation with live-action, juxtaposing black-and-white footage with hyper-saturated color fields, and inserting rapid domino tiles falling on a television screen, the editing utilizes optical counterpoint to force the audience into an active state of intellectual awakening regarding the underlying themes of causality and free will.  Formal Motifs: Chromatic Traffic Lights and the Geometry of the SpiralThe thematic progression of the three runs is meticulously color-coded, utilizing a symbolic matrix that mirrors a standard urban traffic light system:  Figure 4: Chromatic Structural Infographic┌──────────────────────────┐  ┌──────────────────────────┐  ┌──────────────────────────┐
│       RUN 1: RED         │  │       RUN 2: GREEN       │  │       RUN 3: GOLD        │
├──────────────────────────┤  ├──────────────────────────┤  ├──────────────────────────┤
│ • Institutional Refusal  │  │ • Individual Agency      │  │ • Cosmic Hesitation      │
│ • Saturated Cash Bag     │  │ • Armed Bank Robbery     │  │ • Casino Probability     │
│ • Structural STOP        │  │ • Unchecked Will GO      │  │ • Perfect Equilibrium    │
└──────────────────────────┘  └──────────────────────────┘  └──────────────────────────┘
The Chromatic BreakdownThe First Run (Red): Lola seeks institutional assistance from her father at the bank but faces an absolute, patriarchal refusal. The editing relies heavily on sharp, jarring cuts that emphasize her isolation. When Manni and Lola attempt a desperate, uncoordinated supermarket robbery at the end of the run, the bag containing the recovered cash is cast in an ominous, saturated red hue. Red serves as an absolute structural stop command; it denotes chaos, a lack of spiritual alignment, and ultimately results in Lola being shot through the chest by a stray police bullet.  The Second Run (Green): Lola abandons traditional institutional reliance and assumes absolute, aggressive command of her own destiny. She forcefully robs her father’s bank, using a security guard's weapon to secure the capital. The cutting in this sequence is longer and more fluid, tracking her movements with kinetic steadicam shots. The bag containing the stolen currency is a vivid, deep green—the universal signifier for "go". This run represents raw, unchecked human will operating with total disregard for external consequences, which ultimately balance out when Manni is crushed and killed by a passing ambulance.  The Third Run (Yellow / Gold): Lola shifts away from aggressive individual force, choosing instead to slow down, hesitate, and wait to receive an organic sign from the cosmos. She opens herself up to the hidden order of the universe, and the cosmos responds by providing precisely what she requires at an elite casino. Through her visceral, glass-shattering screams at the roulette table, Lola successfully bends the chaotic probability of chance to her absolute will, repeatedly hitting the number 20 until she amasses the needed funds. The currency bag in this final, successful run is a radiant yellow/gold, symbolizing spatial equilibrium, precise timing, and perfect harmony with the causal web.  The Spiral MotifThis structural harmony brings us to the crucial geometric motif running throughout the film: the spiral. Tykwer scatters spirals across the visual landscape of Berlin—from the animated intro sequence where Lola is swallowed by a temporal vortex, to the spiral architecture of the bar near Manni's telephone booth, the winding staircases, pillowcases, and the abstract artwork hanging inside the casino.  As outlined in Darren Aronofsky’s mathematical film Pi, the spiral represents a unified, fractal organizing principle operating across every scale of natural reality, from the golden rectangle of Pythagoras and Da Vinci to tornadoes, DNA strands, and the rotation of the Milky Way galaxy.  Figure 5: The Fractal Spiral Architecture┌─────────────────────────────────┐
│  ┌──────────────┐               │  [Pythagorean Grid Alignment Matrix]
│  │              │               │  
│  │   ┌───┐      │               │  Logarithmic curves track the evolution 
│  │   └───┘      │               │  of the spiral from pure geometry to 
│  └──────────────┘               │  universal structural order.
└─────────────────────────────────┘
The spiral symbolizes a grand, highly structured order that can easily appear terrifying and chaotic to the individual simply because we lack the cognitive capacity to comprehend its immense vastness. This direct visual reference pays homage to Alfred Hitchcock’s Vertigo (1958), a masterpiece centered on characters losing their psychological equilibrium when confronted with mystery, obsessions, and the profound longing for a second chance at rewriting history. By tracking Lola through these recurring spiral environments, Bonnefoy’s editing communicates that while life may appear frantic and out of control, it is actually held within a wider, beautifully organized cosmic design. Chaos is merely order that we have not yet developed the eyes to see.  ConclusionTom Tykwer’s Run Lola Run stands as a benchmark of contemporary avant-garde cinema precisely because its technical execution is completely inseparable from its philosophical depth. Through an aggressive, non-linear editing framework that utilizes parallel realities, video game logic, and rapid photographic montages, the film moves past traditional dramatic structures to map out a complex world of human causality. By treating time not as a continuous line but as an adjustable loop of quantum possibilities, the film demonstrates that our smallest, most routine daily actions carry significant weight. Through its editing choices, Run Lola Run offers an empowering message: we are not merely passive leaves blown about by fate; we are active, conscious designers fully capable of rewriting our paths and shaping our own destinies.  Works CitedAltman, Rick. "Moving Lips: Cinema as Ventriloquism." Yale French Studies, no. 60, 1980, pp. 67-79.  Arnheim, Rudolph. Film as Art. University of California Press, 1971.  Bizzocchi, David. "Lola Runs: Remediation, Immediacy, and Hypermediacy in Tom Tykwer's Digital Fable." Simon Fraser University, 2005, www.sfu.ca/~bizzocch/documents/Lola.pdf.Bordwell, David, and Kristin Thompson. Film Art: An Introduction. 3rd ed., McGraw Hill, 1990.  Carroll, Noël. "Film, Attention, and Communication." The Great Ideas of Today, Encyclopedia Britannica, 1996, pp. 4-49.  Deren, Maya. "Cinematography: The Creative Use of Reality." Film Theory and Criticism: Introductory Readings, edited by Gerald Mast, Marshall Cohen, and Leo Braudy, 4th ed., Oxford University Press, 1992, pp. 59-70.  Eisenstein, Sergei. "A Dialectic Approach to Film Form." Film Theory and Criticism: Introductory Readings, edited by Gerald Mast, Marshall Cohen, and Leo Braudy, 4th ed., Oxford University Press, 1992, pp. 138-154.  Fischer, Lucy. "Film Editing." A Companion to Film Theory, edited by Toby Miller and Robert Stam, Blackwell Publishing Ltd, 2004, pp. 64-81.  Münsterberg, Hugo. The Film: A Psychological Study (The Silent Photoplay in 1916). Dover, 1970.  Run Lola Run (Lola rennt). Directed by Tom Tykwer, performances by Franka Potente and Moritz Bleibtreu, X-Filme Creative Pool / WDR / Arte, 1998.

 """


In [5]:
# gemini generated text:
predict_text(my_text)

[2628 words] "Algorithmic Determinism and the Quantum Aesthetic: How Editing Constructs Mean..."
    LogReg     -> AI    | AI  91.6%  /  human   8.4%
    LinearSVC  -> AI    | AI  62.6%  /  human  37.4%   (uncalibrated approx)



{'LogReg': {'pred': 'AI', 'p_ai': 0.9159361621891278, 'calibrated': True},
 'LinearSVC': {'pred': 'AI', 'p_ai': 0.626316685818292, 'calibrated': False}}

## 5. Classify a list of PDFs — batch comparison table
The 6 PDFs live in the **Downloads** folder as `human_*` / `ai_*` pairs on 3 topics. We list the
paths explicitly; the true label is parsed from the `human_`/`ai_` prefix so we can score
correctness. Because each pair shares a topic, this isolates *authorship* detection from topic.

In [11]:
import re
import pandas as pd
from pypdf import PdfReader

DOWNLOADS = os.path.join(os.path.expanduser('~'), 'Downloads')   # C:/Users/<you>/Downloads
pdf_paths = [
    os.path.join(DOWNLOADS, 'human_01_public_domain.pdf'),
    os.path.join(DOWNLOADS, 'ai_01_public_domain.pdf'),
    os.path.join(DOWNLOADS, 'human_02_aspirin_pharmacology.pdf'),
    os.path.join(DOWNLOADS, 'ai_02_aspirin_pharmacology.pdf'),
    os.path.join(DOWNLOADS, 'human_03_aspirin_history.pdf'),
    os.path.join(DOWNLOADS, 'ai_03_aspirin_history.pdf'),
]

def read_pdf(path):
    reader = PdfReader(path)
    return ' '.join((page.extract_text() or '') for page in reader.pages)

def true_from_name(fn):
    low = fn.lower()
    if low.startswith('ai_'):    return 'AI'
    if low.startswith('human_'): return 'human'
    return '?'

def topic_from_name(fn):
    parts = re.sub(r'\.pdf$', '', fn, flags=re.I).split('_')
    return '_'.join(parts[2:]) if len(parts) > 2 else fn   # drop 'ai_03_' style prefix

rows = []
for path in pdf_paths:
    fn = os.path.basename(path)
    if not os.path.isfile(path):
        rows.append({'file': fn, 'true': true_from_name(fn), 'note': 'FILE NOT FOUND'})
        continue
    text = read_pdf(path)
    words = len(text.split())
    row = {'file': fn, 'topic': topic_from_name(fn), 'words': words, 'true': true_from_name(fn)}
    if words < 10:
        row['note'] = 'no extractable text (scanned image?)'
    else:
        res = predict_text(text, show=False)
        for name, r in res.items():
            row[name] = r['pred']
            row[f'{name}_AI%'] = round(r['p_ai'] * 100, 1)
            row[f'{name}_human%'] = round((1 - r['p_ai']) * 100, 1)
    rows.append(row)

table = pd.DataFrame(rows)
display(table)

# Accuracy per model on the labelled PDFs that had extractable text.
labelled = table[table['true'].isin(['AI', 'human']) & table.get('LogReg', pd.Series(dtype=object)).notna()]
if len(labelled):
    print()
    for name in models:
        hits = int((labelled[name] == labelled['true']).sum())
        print(f'{name:10s}: {hits}/{len(labelled)} correct = {hits/len(labelled):.0%}')
    print('\nWatch the pairs: if human_0X and ai_0X (same topic) get the SAME prediction, the model')
    print('is reading topic/format, not authorship -- the shortcut, on real documents.')

,file,topic,words,true,LogReg,LogReg_AI%,LogReg_human%,LinearSVC,LinearSVC_AI%,LinearSVC_human%
0,human_01_public_domain.pdf,public_domain,621,human,human,14.7,85.3,human,34.4,65.6
1,ai_01_public_domain.pdf,public_domain,461,AI,AI,99.8,0.2,AI,84.3,15.7
2,human_02_aspirin_pharmacology.pdf,aspirin_pharmacology,459,human,human,38.4,61.6,human,36.0,64.0
3,ai_02_aspirin_pharmacology.pdf,aspirin_pharmacology,407,AI,AI,94.7,5.3,AI,64.6,35.4
4,human_03_aspirin_history.pdf,aspirin_history,563,human,AI,97.9,2.1,AI,66.9,33.1
5,ai_03_aspirin_history.pdf,aspirin_history,487,AI,AI,97.1,2.9,AI,66.8,33.2



LogReg    : 5/6 correct = 83%
LinearSVC : 5/6 correct = 83%

Watch the pairs: if human_0X and ai_0X (same topic) get the SAME prediction, the model
is reading topic/format, not authorship -- the shortcut, on real documents.


## Caveat, stated plainly
A supervised model's eval score is only trustworthy **on data drawn from the same distribution as
its training set**. This baseline is near-perfect on the PERSUADE-vs-AI-essay data it was built
on, but the generalization test above shows the score **does not carry over** to arbitrary
documents — it keys on dataset artifacts, not universal signals of machine-generated text. Report
the baseline honestly with this limitation, and treat the Week-2 Transformer comparison as
"same-dataset", not "real-world AI detection".